# MediLink TOON RAG — End-to-End Evaluation (Colab)

Runs the end-to-end TOON evaluation (`eval_endtoend.py`) on a Colab GPU to measure the **post-fix T3 hallucination rate** after the grounding-gate + prompt rewrite.

**Before you start:**
1. `Runtime → Change runtime type → Hardware accelerator → GPU` → Save.
2. Click the **key icon** (left sidebar) and add three secrets with *Notebook access* ON:
   - `GROQ_API_KEY`
   - `NEXT_PUBLIC_SUPABASE_URL`  →  `https://icntpbdznkfajnieyrjq.supabase.co`
   - `NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY`

Run the cells top-to-bottom.

## 1. Clone the repo and install dependencies

In [ ]:
import os
if not os.path.isdir('MediLink-RAG'):
    !git clone https://github.com/fareshassan22/MediLink-RAG.git
%cd MediLink-RAG
!pip install -q -r requirements.txt

## 2. Load API keys from Colab Secrets into the environment
Keys are read from the secret store — they never appear in the notebook.

In [ ]:
import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY'] = userdata.get('NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY')
try:
    os.environ['NEXT_PUBLIC_SUPABASE_URL'] = userdata.get('NEXT_PUBLIC_SUPABASE_URL')
except Exception:
    os.environ['NEXT_PUBLIC_SUPABASE_URL'] = 'https://icntpbdznkfajnieyrjq.supabase.co'

# Colab has a single GPU (device 0). The script defaults to device 7, so override it here.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

assert os.environ['GROQ_API_KEY'].startswith('gsk_'), 'GROQ_API_KEY looks wrong — check the secret value'
print('Secrets loaded. Supabase URL:', os.environ['NEXT_PUBLIC_SUPABASE_URL'])

## 3. Verify the GPU is visible (should print a real GPU, not hang)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 4. Quick connectivity check (Supabase + Groq)
Confirms the keys work before committing to the full run.

In [ ]:
import os, requests, time
url = os.environ['NEXT_PUBLIC_SUPABASE_URL']
key = os.environ['NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY']
h = {'apikey': key, 'Authorization': f'Bearer {key}'}
t = time.time()
r = requests.get(f'{url}/rest/v1/appointments?patient_id=eq.151&select=id&limit=1', headers=h, timeout=10)
print('Supabase:', r.status_code, '(%.1fs)' % (time.time() - t), r.text[:80])

from groq import Groq
t = time.time()
c = Groq(api_key=os.environ['GROQ_API_KEY'])
resp = c.chat.completions.create(model='llama-3.1-8b-instant', messages=[{'role': 'user', 'content': 'say ok'}], max_tokens=5)
print('Groq:', resp.choices[0].message.content, '(%.1fs)' % (time.time() - t))

## 5. Run the end-to-end evaluation
100 queries, serial (Groq free-tier rate limit ~30s/query). **Resumable** — if Colab disconnects, just re-run this cell and it continues from the checkpoint.

First run downloads the embedder + reranker (~2.5 GB), so the first query is slow.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python eval_endtoend.py

## 6. Show the summary (overall + per-tier grounding / hallucination)

In [ ]:
import glob, pandas as pd
latest = sorted(glob.glob('results/endtoend_summary_*.csv'))[-1]
print('File:', latest)
pd.read_csv(latest)

## 7. Download the results
Results are not auto-committed to GitHub — download them to keep.

In [ ]:
from google.colab import files
import glob
files.download(sorted(glob.glob('results/endtoend_summary_*.csv'))[-1])
files.download('results/endtoend_checkpoint.jsonl')